# RouteZone — Notebook 05 : MLflow, Tuning & Model Registry
#### Meriem Abdelouahed | Formation Dev IA — Simplon x Microsoft
---
## Sommaire
1. [MLflow — contexte et utilité](#1-mlflow)
2. [Imports et configuration](#2-imports)
3. [Chargement, binarisation et split](#3-data)
4. [Expérience 1 — Tuning manuel](#4-tuning-manuel)
5. [Expérience 2 — GridSearch](#5-gridsearch)
6. [Expérience 3 — Optuna F1 macro](#6-optuna-f1)
7. [Expérience 4 — Optuna Recall GRAVE](#7-optuna-recall)
8. [Comparaison des 4 méthodes](#8-comparaison)
9. [Model Registry](#9-registry)
10. [Bilan](#10-bilan)

---
> **Objectif :**
> Le notebook_04 a sélectionné LightGBM comme meilleur algorithme (meilleur Recall GRAVE).
> Ce notebook cherche les **meilleurs paramètres** pour LightGBM via 3 méthodes de tuning,
> toutes trackées dans MLflow pour pouvoir les comparer et les reproduire.
>
> **Pourquoi optimiser les paramètres ?**
> Un algorithme avec des paramètres par défaut n'est pas optimal.
> Le tuning consiste à trouver la combinaison de paramètres qui maximise
> la métrique choisie — ici le Recall GRAVE ou le F1 macro.

<a id='1-mlflow'></a>
## 1. MLflow — contexte et utilité
---

### Problématique

Sans système de traçabilité, chaque nouvelle exécution du notebook écrase les résultats précédents.
Il devient impossible de comparer les configurations testées ou de retrouver les paramètres
ayant produit les meilleures performances.

### Solution : MLflow

MLflow est un outil open source de gestion du cycle de vie des modèles de machine learning.
Il enregistre automatiquement à chaque entraînement :

| Ce qu'il enregistre | Exemple |
|---|---|
| **Paramètres** | n_estimators=200, max_depth=8 |
| **Métriques** | F1=0.716, AUC-ROC=0.866, Recall GRAVE=0.805 |
| **Modèle** | Le fichier sérialisé du modèle entraîné |
| **Artefacts** | Graphiques, matrices de confusion |

### Les 4 composants MLflow

- **Tracking** → enregistre chaque expérience (utilisé dans ce notebook)
- **Projects** → package le code pour le reproduire
- **Models** → format standard pour sauvegarder les modèles
- **Registry** → catalogue versionné des modèles (section 9)

### Lancement de l'interface

```bash
# Dans le terminal :
mlflow ui
# Interface accessible sur : http://127.0.0.1:5000
```

> **Observations :**
> - Binarisation : 82.6% Pas grave, 17.4% Grave — déséquilibre géré par `class_weight='balanced'`
> - Split stratifié : même proportion GRAVE dans le train (17.4%), la validation et le test
> - Résultats directement comparables au notebook_04 : mêmes données, même split

<a id='4-tuning-manuel'></a>
## 4. Expérience 1 — Tuning manuel
---

### Principe

Le tuning manuel consiste à tester quelques configurations choisies à la main
en faisant varier les hyperparamètres clés.
C'est la méthode la plus simple — elle donne un premier aperçu
de la sensibilité du modèle à chaque paramètre.

### Pourquoi LightGBM ?

LightGBM est le modèle retenu dans le notebook_04 car il présente
le meilleur **Recall GRAVE** (79.6%) parmi les 3 algorithmes testés.
Dans un contexte de sécurité routière, rater un accident grave est plus dangereux
qu'une fausse alarme — le recall est donc le critère prioritaire.

### Suivi MLflow

Chaque configuration est enregistrée comme un **run distinct** dans MLflow.
Tous les runs sont consultables et comparables via `mlflow ui`.

In [1]:
mlflow.set_experiment("RouteZone_Tuning_Manuel")

tuning_configs = [
    {"nom": "Config-1-rapide",      "n_estimators": 200, "max_depth": 8,  "learning_rate": 0.05},
    {"nom": "Config-2-equilibree",  "n_estimators": 300, "max_depth": 10, "learning_rate": 0.03},
    {"nom": "Config-3-profonde",    "n_estimators": 500, "max_depth": 12, "learning_rate": 0.01},
]

tuning_results = []

for config in tuning_configs:
    nom = config.pop("nom")

    with mlflow.start_run(run_name=nom):
        params = {
            **config,
            "subsample": 0.8, "colsample_bytree": 0.8,
            "class_weight": "balanced", "random_state": 42,
            "n_jobs": -1, "verbose": -1
        }
        mlflow.log_params(params)

        model = LGBMClassifier(**params)
        model.fit(X_train, y_train)

        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        f1m     = f1_score(y_test, y_pred, average='macro')
        auc     = roc_auc_score(y_test, y_proba)
        recall  = recall_score(y_test, y_pred, pos_label=1)

        mlflow.log_metric("f1_macro_test", f1m)
        mlflow.log_metric("auc_roc",       auc)
        mlflow.log_metric("recall_grave",  recall)
        mlflow.sklearn.log_model(model, nom)

        tuning_results.append({
            "Config": nom, "F1 macro": round(f1m, 4),
            "AUC-ROC": round(auc, 4), "Recall GRAVE": round(recall, 4)
        })
        print(f'{nom} → F1={f1m:.4f} | AUC={auc:.4f} | Recall={recall:.4f}')

df_tuning = pd.DataFrame(tuning_results).set_index("Config")
print()
print(df_tuning.to_string())
print(f'\n→ Meilleur Recall GRAVE : {df_tuning["Recall GRAVE"].idxmax()}')

NameError: name 'mlflow' is not defined

> **Observations :**
> - **Config-1-rapide** (200 arbres, lr=0.05) : F1=0.684 | Recall GRAVE=0.798
> - **Config-2-equilibree** (300 arbres, lr=0.03) : F1=0.683 | Recall GRAVE=**0.800** ← meilleure
> - **Config-3-profonde** (500 arbres, lr=0.01) : F1=0.676 | Recall GRAVE=0.796 ← moins bonne
> - Les 3 configurations donnent des résultats très proches — l'espace de recherche manuel est restreint
> - La Config-3 plus profonde et lente est légèrement moins performante : trop de complexité pour ce dataset
> - **Limite :** 3 points testés dans un espace de paramètres très large — GridSearch et Optuna vont l'explorer plus rigoureusement

<a id='5-gridsearch'></a>
## 5. Expérience 2 — GridSearch (recherche exhaustive)
---

### Principe

GridSearch teste **toutes les combinaisons possibles** d'une grille de paramètres prédéfinie.
Avec cross-validation à 3 folds, chaque combinaison est évaluée 3 fois sur des
sous-ensembles différents du jeu d'entraînement — estimation plus robuste des performances.

**Nombre d'entraînements** = 8 combinaisons × 3 folds = **24 entraînements au total**

| | GridSearch |
|---|---|
| Avantage | Exhaustif — garantit la meilleure combinaison sur la grille définie |
| Limite | Limité à la grille fixe — si la meilleure config est entre deux valeurs, elle est ratée |

Chaque combinaison = un run MLflow distinct.

In [ ]:
from sklearn.model_selection import GridSearchCV

mlflow.set_experiment("RouteZone_GridSearch")

# 2 × 2 × 2 = 8 combinaisons × 3 folds = 24 entraînements
param_grid = {
    "n_estimators":  [200, 300],
    "max_depth":     [8, 12],
    "learning_rate": [0.05, 0.01],
}

print(f'Nombre de combinaisons : {2*2*2} × 3 folds = {2*2*2*3} entraînements')

base_model = LGBMClassifier(
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced', random_state=42,
    n_jobs=-1, verbose=-1
)

# scoring='f1_macro' → optimise sur le F1 macro
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f'\n✓ GridSearch terminé')
print(f'Meilleurs paramètres : {grid_search.best_params_}')
print(f'Meilleur F1 macro CV : {grid_search.best_score_:.4f}')

In [ ]:
# Logger chaque combinaison dans MLflow
grid_results = []

for i, params in enumerate(grid_search.cv_results_['params']):
    with mlflow.start_run(run_name=f"GridSearch-combo-{i+1}"):

        mlflow.log_params(params)
        mlflow.log_metric("f1_macro_cv", grid_search.cv_results_['mean_test_score'][i])

        model_gs = LGBMClassifier(
            **params, subsample=0.8, colsample_bytree=0.8,
            class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
        )
        model_gs.fit(X_train, y_train)

        y_pred  = model_gs.predict(X_test)
        y_proba = model_gs.predict_proba(X_test)[:, 1]
        f1m     = f1_score(y_test, y_pred, average='macro')
        auc     = roc_auc_score(y_test, y_proba)
        recall  = recall_score(y_test, y_pred, pos_label=1)

        mlflow.log_metric("f1_macro_test", f1m)
        mlflow.log_metric("auc_roc",       auc)
        mlflow.log_metric("recall_grave",  recall)

        grid_results.append({
            **params,
            "F1 macro CV": round(grid_search.cv_results_['mean_test_score'][i], 4),
            "F1 macro test": round(f1m, 4),
            "AUC-ROC": round(auc, 4),
            "Recall GRAVE": round(recall, 4)
        })

df_grid = pd.DataFrame(grid_results).sort_values('Recall GRAVE', ascending=False)
print('Résultats GridSearch (triés par Recall GRAVE) :')
print(df_grid.to_string(index=False))

> **Observations :**
> - Meilleurs paramètres (F1 macro CV) : learning_rate=0.05, max_depth=8, n_estimators=300
> - Meilleur Recall GRAVE : 0.799 (learning_rate=0.05, max_depth=12, n_estimators=200)
> - Résultats très proches du tuning manuel — la grille était déjà bien calibrée
> - Les configs avec learning_rate=0.01 sont nettement moins bonnes (Recall < 0.793)
> - **Constat :** GridSearch confirme le tuning manuel mais ne le dépasse pas —
>   la grille est trop petite pour explorer l'espace complet des paramètres
> - Optuna va explorer un espace bien plus large de façon intelligente

<a id='6-optuna-f1'></a>
## 6. Expérience 3 — Optuna (optimisation F1 macro)
---

### Principe

Optuna utilise la **recherche bayésienne** : il apprend de chaque essai
pour orienter les suivants vers les zones prometteuses de l'espace de recherche.
Contrairement à GridSearch qui teste une grille fixe, Optuna explore des valeurs continues.

| | GridSearch | Optuna |
|---|---|---|
| Méthode | Exhaustif sur grille fixe | Bayésien, adaptatif |
| Espace exploré | Petit (grille prédéfinie) | Large (plages continues) |
| Nb d'essais | 8 × 3 = 24 | 50 trials |

### Scoring utilisé : `f1_macro`

Cette expérience optimise le F1 macro. L'expérience suivante testera `scoring='recall'`
pour démontrer l'impact du choix du scoring sur les paramètres trouvés.

In [ ]:
mlflow.set_experiment("RouteZone_Optuna")

def objective_f1(trial):
    params = {
        "n_estimators":  trial.suggest_int(  "n_estimators",  100, 500),
        "max_depth":     trial.suggest_int(  "max_depth",     4,   15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves":    trial.suggest_int(  "num_leaves",    20,  100),
        "subsample":     trial.suggest_float("subsample",     0.6,  1.0),
        "class_weight":  "balanced",
        "random_state":  42, "n_jobs": -1, "verbose": -1
    }
    model = LGBMClassifier(**params)
    # scoring='f1_macro' → Optuna cherche les paramètres qui maximisent le F1 macro
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='f1_macro')
    return scores.mean()

print('Optuna — 50 trials — scoring=f1_macro...')

with mlflow.start_run(run_name="Optuna-f1macro-study"):
    study_f1 = optuna.create_study(direction="maximize")
    study_f1.optimize(objective_f1, n_trials=50, show_progress_bar=True)

    best_params_f1 = study_f1.best_params
    mlflow.log_params(best_params_f1)
    mlflow.log_metric("best_f1_cv", study_f1.best_value)
    mlflow.log_param("scoring", "f1_macro")

print(f'\n✓ Terminé | Meilleur F1 CV : {study_f1.best_value:.4f}')
print(f'Paramètres : {best_params_f1}')

In [ ]:
# Évaluation du meilleur modèle Optuna F1 sur le test
with mlflow.start_run(run_name="Optuna-f1macro-best"):
    model_optuna_f1 = LGBMClassifier(
        **best_params_f1, class_weight="balanced",
        random_state=42, n_jobs=-1, verbose=-1
    )
    model_optuna_f1.fit(X_train, y_train)

    y_pred_f1  = model_optuna_f1.predict(X_test)
    y_proba_f1 = model_optuna_f1.predict_proba(X_test)[:, 1]
    f1m_f1     = f1_score(y_test, y_pred_f1, average='macro')
    auc_f1     = roc_auc_score(y_test, y_proba_f1)
    recall_f1  = recall_score(y_test, y_pred_f1, pos_label=1)

    mlflow.log_params({**best_params_f1, "scoring": "f1_macro"})
    mlflow.log_metric("f1_macro_test", f1m_f1)
    mlflow.log_metric("auc_roc",       auc_f1)
    mlflow.log_metric("recall_grave",  recall_f1)
    mlflow.sklearn.log_model(model_optuna_f1, "optuna-f1macro-best")

    print(f'Optuna F1 macro — résultats test :')
    print(f'  F1 macro     : {f1m_f1:.4f}')
    print(f'  AUC-ROC      : {auc_f1:.4f}')
    print(f'  Recall GRAVE : {recall_f1:.4f}')
    print()
    print(classification_report(y_test, y_pred_f1, target_names=['Pas grave', 'Grave']))

> **Observations :**
> - Meilleurs paramètres : n_estimators=481, max_depth=14, learning_rate=0.144, num_leaves=89
> - **F1 macro : 0.716** — nettement supérieur au tuning manuel (0.684) et GridSearch (0.687)
> - **AUC-ROC : 0.866** — meilleur de toutes les expériences précédentes
> - **Recall GRAVE : 0.768** — inférieur au tuning manuel (0.800)
> - **Explication :** Optuna optimisait `scoring='f1_macro'`.
>   Il a trouvé les paramètres qui maximisent le F1, au détriment du recall.
>   Ce résultat illustre directement l'impact du choix du scoring dans Optuna.
> - **Hypothèse à tester :** si on change `scoring='recall'`, le Recall GRAVE devrait augmenter

<a id='7-optuna-recall'></a>
## 7. Expérience 4 — Optuna (optimisation Recall GRAVE)
---

### Pourquoi cette expérience ?

L'expérience précédente a montré qu'Optuna avec `scoring='f1_macro'` améliore le F1
mais réduit le Recall GRAVE par rapport au tuning manuel.

L'hypothèse testée ici : **si on demande à Optuna de maximiser le Recall GRAVE,
obtient-on un meilleur résultat que le tuning manuel ?**

C'est précisément l'utilité de MLflow : formuler une hypothèse, l'enregistrer,
comparer avec les expériences précédentes, et décider objectivement.

### Seul changement vs l'expérience précédente

```python
# Expérience 3 : scoring='f1_macro'
# Expérience 4 : scoring='recall'  ← unique changement
scores = cross_val_score(model, X_train, y_train, cv=3, scoring='recall')
```

In [ ]:
def objective_recall(trial):
    params = {
        "n_estimators":  trial.suggest_int(  "n_estimators",  100, 500),
        "max_depth":     trial.suggest_int(  "max_depth",     4,   15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves":    trial.suggest_int(  "num_leaves",    20,  100),
        "subsample":     trial.suggest_float("subsample",     0.6,  1.0),
        "class_weight":  "balanced",
        "random_state":  42, "n_jobs": -1, "verbose": -1
    }
    model = LGBMClassifier(**params)
    # scoring='recall' → Optuna cherche les paramètres qui maximisent le Recall GRAVE
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='recall')
    return scores.mean()

print('Optuna — 50 trials — scoring=recall...')

with mlflow.start_run(run_name="Optuna-recall-study"):
    study_recall = optuna.create_study(direction="maximize")
    study_recall.optimize(objective_recall, n_trials=50, show_progress_bar=True)

    best_params_recall = study_recall.best_params
    mlflow.log_params(best_params_recall)
    mlflow.log_metric("best_recall_cv", study_recall.best_value)
    mlflow.log_param("scoring", "recall")

print(f'\n✓ Terminé | Meilleur Recall CV : {study_recall.best_value:.4f}')
print(f'Paramètres : {best_params_recall}')

In [ ]:
# Évaluation du meilleur modèle Optuna Recall sur le test
with mlflow.start_run(run_name="Optuna-recall-best"):
    model_optuna_recall = LGBMClassifier(
        **best_params_recall, class_weight="balanced",
        random_state=42, n_jobs=-1, verbose=-1
    )
    model_optuna_recall.fit(X_train, y_train)

    y_pred_recall  = model_optuna_recall.predict(X_test)
    y_proba_recall = model_optuna_recall.predict_proba(X_test)[:, 1]
    f1m_recall     = f1_score(y_test, y_pred_recall, average='macro')
    auc_recall     = roc_auc_score(y_test, y_proba_recall)
    recall_recall  = recall_score(y_test, y_pred_recall, pos_label=1)

    mlflow.log_params({**best_params_recall, "scoring": "recall"})
    mlflow.log_metric("f1_macro_test", f1m_recall)
    mlflow.log_metric("auc_roc",       auc_recall)
    mlflow.log_metric("recall_grave",  recall_recall)
    mlflow.sklearn.log_model(model_optuna_recall, "optuna-recall-best")

    print(f'Optuna Recall GRAVE — résultats test :')
    print(f'  F1 macro     : {f1m_recall:.4f}')
    print(f'  AUC-ROC      : {auc_recall:.4f}')
    print(f'  Recall GRAVE : {recall_recall:.4f}')
    print()
    print(classification_report(y_test, y_pred_recall, target_names=['Pas grave', 'Grave']))

> **Observations :**
> - Meilleurs paramètres : n_estimators=170, max_depth=15, learning_rate=0.019, num_leaves=94
> - **Recall GRAVE : 0.805** — meilleur de toutes les expériences, dépasse le tuning manuel (0.800)
> - **F1 macro : 0.676** — inférieur à Optuna F1 macro (0.716) — trade-off attendu
> - **AUC-ROC : 0.850** — légèrement inférieur à Optuna F1 macro (0.866)
> - **Confirmation de l'hypothèse :** `scoring='recall'` produit bien un meilleur recall
>   au prix d'un F1 macro plus faible
> - **Trade-off recall / precision :** plus on détecte d'accidents graves (recall élevé),
>   plus on génère de fausses alarmes (precision GRAVE = 0.39 vs 0.45 pour Optuna F1)
>   Ce compromis est inhérent à tout problème de classification déséquilibré

<a id='8-comparaison'></a>
## 8. Comparaison des 4 méthodes de tuning
---
Synthèse complète des 4 expériences avec les résultats obtenus.

In [ ]:
# Récupérer le meilleur de chaque méthode
meilleur_tuning  = df_tuning.sort_values('Recall GRAVE', ascending=False).iloc[0]
meilleur_grid    = df_grid.sort_values('Recall GRAVE', ascending=False).iloc[0]

comparaison = pd.DataFrame({
    'Méthode':       ['Tuning manuel', 'GridSearch', 'Optuna F1 macro', 'Optuna Recall'],
    'Nb essais':     [3, len(grid_results), 50, 50],
    'F1 macro test': [
        meilleur_tuning['F1 macro'],
        meilleur_grid['F1 macro test'],
        round(f1m_f1, 4),
        round(f1m_recall, 4)
    ],
    'AUC-ROC':       [
        meilleur_tuning['AUC-ROC'],
        meilleur_grid['AUC-ROC'],
        round(auc_f1, 4),
        round(auc_recall, 4)
    ],
    'Recall GRAVE':  [
        meilleur_tuning['Recall GRAVE'],
        meilleur_grid['Recall GRAVE'],
        round(recall_f1, 4),
        round(recall_recall, 4)
    ],
}).set_index('Méthode')

print('=' * 70)
print('  COMPARAISON DES 4 MÉTHODES DE TUNING')
print('=' * 70)
print(comparaison.to_string())
print('=' * 70)
print(f'\n→ Meilleur F1 macro  : {comparaison["F1 macro test"].idxmax()}')
print(f'→ Meilleur Recall GRAVE : {comparaison["Recall GRAVE"].idxmax()}')

In [ ]:
# Graphique comparatif
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Comparaison des 4 méthodes de tuning', fontsize=13, fontweight='bold')

metriques = ['F1 macro test', 'AUC-ROC', 'Recall GRAVE']
couleurs  = ['#2980b9', '#e67e22', '#27ae60', '#8e44ad']

for idx, metrique in enumerate(metriques):
    valeurs = comparaison[metrique].values
    barres  = axes[idx].bar(
        comparaison.index, valeurs,
        color=couleurs, alpha=0.85, edgecolor='white'
    )
    axes[idx].set_title(metrique, fontsize=11, fontweight='bold')
    axes[idx].set_ylim(min(valeurs) - 0.05, max(valeurs) + 0.05)
    axes[idx].tick_params(axis='x', rotation=20)
    for bar, val in zip(barres, valeurs):
        axes[idx].text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', fontsize=9, fontweight='bold'
        )

plt.tight_layout()
plt.savefig('./visualisations/viz_nb05_comparaison_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

> **Observations :**
>
> | Méthode | Nb essais | F1 macro | AUC-ROC | Recall GRAVE |
> |---|---|---|---|---|
> | Tuning manuel | 3 | 0.683 | 0.853 | 0.800 |
> | GridSearch | 8 | 0.683 | 0.854 | 0.799 |
> | Optuna F1 macro | 50 | **0.716** | **0.866** | 0.768 |
> | Optuna Recall | 50 | 0.676 | 0.850 | **0.805** |
>
> - Tuning manuel et GridSearch donnent des résultats quasi identiques — la grille était bien calibrée
> - Optuna F1 macro améliore le F1 (+5%) et l'AUC-ROC mais réduit le Recall GRAVE
> - Optuna Recall obtient le meilleur Recall GRAVE (0.805) — dépasse le tuning manuel
> - Le scoring dans Optuna oriente concrètement les paramètres trouvés
>
> **Modèle retenu : Optuna Recall** — critère prioritaire = Recall GRAVE.
> Détecter 8 accidents graves sur 10 est plus important qu'un score global légèrement meilleur
> dans un contexte de sécurité routière.

<a id='9-registry'></a>
## 9. Model Registry
---

### Rôle du Model Registry

Le Model Registry MLflow est un catalogue centralisé des modèles versionnés.
Il gère le **cycle de vie** d'un modèle :

| Statut | Signification |
|---|---|
| **None** | Enregistré, pas encore validé |
| **Staging** | En phase de test |
| **Production** | Validé, utilisé par la FastAPI |
| **Archived** | Ancienne version, plus utilisée |

Chaque nouveau modèle entraîné avec de meilleures performances crée automatiquement
une nouvelle version sans écraser les précédentes.

In [ ]:
# On sélectionne le meilleur modèle selon le Recall GRAVE
# (critère médical prioritaire pour RouteZone)
metriques_comparaison = {
    'Tuning manuel':    (None, meilleur_tuning['Recall GRAVE']),
    'Optuna F1 macro':  (model_optuna_f1, recall_f1),
    'Optuna Recall':    (model_optuna_recall, recall_recall),
}

meilleur_nom    = max(metriques_comparaison, key=lambda k: metriques_comparaison[k][1])
meilleur_modele = metriques_comparaison[meilleur_nom][0]
meilleur_recall = metriques_comparaison[meilleur_nom][1]

print(f'Meilleur modèle (Recall GRAVE) : {meilleur_nom}')
print(f'Recall GRAVE : {meilleur_recall:.4f}')
print()

# Sauvegarde en .pkl pour la FastAPI IA
if meilleur_modele is not None:
    joblib.dump(meilleur_modele, MODELS_DIR / 'best_model.pkl')
    joblib.dump(features, MODELS_DIR / 'features.pkl')
    joblib.dump({0: 'Pas grave', 1: 'Grave'}, MODELS_DIR / 'class_mapping.pkl')
    print(f'✓ Modèle sauvegardé → {MODELS_DIR}/best_model.pkl')
    print(f'✓ Features          → {MODELS_DIR}/features.pkl')
    print(f'✓ Mapping           → {MODELS_DIR}/class_mapping.pkl')
else:
    print('Le meilleur modèle est le tuning manuel — charger depuis MLflow')

> **Observations :**
> - **Modèle retenu :** Optuna Recall — Recall GRAVE = 0.805 (meilleur de toutes les expériences)
> - 3 fichiers sauvegardés pour la FastAPI IA :
>   - `best_model.pkl` → le modèle fait les prédictions (retourne 0 ou 1)
>   - `features.pkl` → liste des features dans le bon ordre — la FastAPI doit envoyer
>     exactement ces colonnes dans le même ordre qu'à l'entraînement
>   - `class_mapping.pkl` → convertit 0→Pas grave, 1→Grave pour une réponse lisible
> - Dans l'interface MLflow (onglet **Models**), le statut peut être changé vers **Production**
> - En production, un nouveau modèle crée une version 2 sans supprimer la version 1

<a id='10-bilan'></a>
## 10. Bilan

---

## Bilan du notebook

### Ce qui a été réalisé

**Configuration MLflow** — serveur local avec backend SQLite.
Interface accessible via `mlflow ui` → http://127.0.0.1:5000

**4 expériences de tuning sur LightGBM, toutes trackées dans MLflow :**

| Méthode | Nb essais | F1 macro | AUC-ROC | Recall GRAVE |
|---|---|---|---|---|
| Tuning manuel | 3 | 0.683 | 0.853 | 0.800 |
| GridSearch | 8 | 0.683 | 0.854 | 0.799 |
| Optuna F1 macro | 50 | **0.716** | **0.866** | 0.768 |
| Optuna Recall | 50 | 0.676 | 0.850 | **0.805** |

**Modèle retenu :** Optuna Recall — meilleur Recall GRAVE (0.805).
Sauvegardé dans `models/best_model.pkl` pour la FastAPI IA.

### Enseignement principal

Le choix du `scoring` dans Optuna oriente directement les paramètres vers la métrique optimisée.
Sans MLflow, il serait impossible de comparer ces 4 expériences objectivement.

### Le trade-off recall / precision

| Modèle | Recall GRAVE | Precision GRAVE | Interprétation |
|---|---|---|---|
| Optuna F1 macro | 0.768 | 0.45 | Moins de fausses alarmes, plus d'accidents graves ratés |
| Optuna Recall | 0.805 | 0.39 | Plus de fausses alarmes, moins d'accidents graves ratés |

Dans un contexte de sécurité routière, envoyer les secours inutilement
est préférable à ne pas les envoyer quand c'est nécessaire.
Le modèle Optuna Recall est donc le plus adapté à la problématique.

### Prochaine étape

Le modèle sauvegardé dans `models/best_model.pkl` sera exposé via
une **FastAPI IA** avec un endpoint `/predict` (compétence C9).